## CELL INIT — запускати після кожного перезапуску сесії

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install adversarial-robustness-toolbox xgboost -q

import os, json, warnings, gc
from collections import defaultdict
import numpy as np, pandas as pd
import joblib, torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, f1_score
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from art.attacks.evasion import (FastGradientMethod,
    ProjectedGradientDescent, CarliniL2Method)
from art.estimators.classification import PyTorchClassifier
warnings.filterwarnings('ignore')

BASE       = '/content/drive/MyDrive/adversarial_iot_paper'
SEC6_DIR   = os.path.join(BASE, 'results', 'section6_v2')
SEC7_DIR   = os.path.join(BASE, 'results', 'section7_v2')
MODELS_DIR = os.path.join(BASE, 'models_v2')
os.makedirs(SEC7_DIR, exist_ok=True)

SEED = 42; np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
EPS_LIST   = [0.01, 0.05, 0.1]
CW_SAMPLES = 200
CW_ITERS   = 500
N_AL       = 8   # aligned features for cross-dataset

DS_TAGS = {
    'CIC-IDS 2017'       : 'CIC-IDS_2017',
    'UNSW-NB15'          : 'UNSW-NB15',
    'Gotham IoT 2025'    : 'Gotham_IoT_2025',
    'CIC-YNU-IoTMal 2026': 'CIC-YNU-IoTMal_2026',
}

class CNN1D(nn.Module):
    def __init__(self, n_features, n_classes):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(1,128,3,padding=1), nn.BatchNorm1d(128),
            nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(128,64,3,padding=1), nn.BatchNorm1d(64),
            nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(64,32,3,padding=1), nn.BatchNorm1d(32),
            nn.ReLU(), nn.AdaptiveAvgPool1d(1),
        )
        self.head = nn.Sequential(
            nn.Flatten(), nn.Linear(32,128),
            nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, n_classes)
        )
    def forward(self, x): return self.head(self.conv(x.unsqueeze(1)))

class TorchMLP(nn.Module):
    def __init__(self, n_feat, n_cls):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_feat,256), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(256,128),   nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128,64),    nn.ReLU(),
            nn.Linear(64, n_cls)
        )
    def forward(self, x): return self.net(x)

def train_torch_mlp(X_tr, y_tr, n_feat, n_cls,
                    epochs=50, batch=256, lr=0.001, patience=5):
    model = TorchMLP(n_feat, n_cls).to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=lr)
    crit  = nn.CrossEntropyLoss()
    dl    = DataLoader(TensorDataset(
                torch.FloatTensor(X_tr), torch.LongTensor(y_tr)),
                batch_size=batch, shuffle=True)
    best, no_imp = float('inf'), 0
    for epoch in range(epochs):
        model.train(); total = 0
        for xb,yb in dl:
            xb,yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(); loss = crit(model(xb),yb)
            loss.backward(); opt.step(); total += loss.item()
        avg = total/len(dl)
        if avg < best-1e-4: best=avg; no_imp=0
        else: no_imp+=1
        if no_imp >= patience: break
    model.eval(); return model

def make_art_clf(model, n_feat, n_cls):
    opt = torch.optim.Adam(model.parameters(), lr=0.001)
    return PyTorchClassifier(
        model=model, loss=nn.CrossEntropyLoss(), optimizer=opt,
        input_shape=(n_feat,), nb_classes=n_cls,
        clip_values=(0.0, 1.0),
        device_type='gpu' if DEVICE=='cuda' else 'cpu')

def predict_batched(model, X, batch=4096):
    model.eval(); all_p = []
    with torch.no_grad():
        for i in range(0, len(X), batch):
            xb = torch.FloatTensor(X[i:i+batch]).to(DEVICE)
            all_p.append(torch.argmax(model(xb),1).cpu().numpy())
            del xb
    if DEVICE=='cuda': torch.cuda.empty_cache()
    return np.concatenate(all_p)

# Load DATA
DATA = {}
for ds_name, tag in DS_TAGS.items():
    xtr = np.load(os.path.join(SEC6_DIR, f'X_tr_{tag}.npy'))
    xte = np.load(os.path.join(SEC6_DIR, f'X_te_{tag}.npy'))
    ytr = np.load(os.path.join(SEC6_DIR, f'y_tr_{tag}.npy'))
    yte = np.load(os.path.join(SEC6_DIR, f'y_te_{tag}.npy'))
    n_cls = len(np.unique(ytr))
    DATA[ds_name] = (xtr, xte, ytr, yte, n_cls)
    print(f'✓ DATA/{ds_name}: te={len(xte):,} cls={n_cls} feat={xte.shape[1]}')

# Load sklearn models
MODELS = {}
for ds_name, tag in DS_TAGS.items():
    MODELS[ds_name] = {}
    for mn, key in [('rf','RF'),('xgb','XGB'),('mlp','MLP')]:
        p = os.path.join(MODELS_DIR, f'{mn}_{tag}.joblib')
        if os.path.exists(p):
            MODELS[ds_name][key] = joblib.load(p)
    pt  = os.path.join(MODELS_DIR, f'cnn_{tag}.pt')
    cfg = os.path.join(MODELS_DIR, f'cnn_{tag}_cfg.json')
    if os.path.exists(pt):
        with open(cfg) as f: c = json.load(f)
        m = CNN1D(c['n_features'], c['n_classes']).to(DEVICE)
        m.load_state_dict(torch.load(pt, map_location=DEVICE)); m.eval()
        MODELS[ds_name]['CNN'] = m
    print(f'✓ MODELS/{ds_name}: {list(MODELS[ds_name].keys())}')

print(f'\n✓ INIT complete | Device: {DEVICE}')

## CELL STATUS — перевірка збережених результатів

In [ ]:
print('=== Section 7 v2 STATUS ===')
# Cross-arch: 144 (не 180) — 2 src × 3 tgt × 2 atk × 3 eps × 4 ds
expected = {
    'wb_results_v2.json'     : 120,
    'ca_results_v2.json'     : 144,
    'cd_results_v2.json'     :  36,
    'stat_results_v2.json'   :   2,
    'defense_results_v2.json':  16,
}
for fname, exp in expected.items():
    p = os.path.join(SEC7_DIR, fname)
    if os.path.exists(p):
        data = json.load(open(p))
        valid = sum(1 for v in data.values() if 'error' not in v)
        kb = os.path.getsize(p)/1024
        icon = '✓' if valid >= exp else '⚠'
        print(f'  {icon}  {fname:<35} {kb:>6.0f} KB  ({valid}/{exp})')
    else:
        print(f'  —  {fname}')

## CELL 5 — White-box attacks (120 combinations)
C&W: 200 samples, 500 iters (~5 min/combo на A100)

In [ ]:
WB_PATH = os.path.join(SEC7_DIR, 'wb_results_v2.json')
wb = json.load(open(WB_PATH)) if os.path.exists(WB_PATH) else {}
valid = sum(1 for v in wb.values() if 'error' not in v)
print(f'White-box: {valid}/120 already computed.')

# TorchMLP — потрібен для ART (sklearn MLP не має градієнтів)
TORCH_MLPS = {}
for ds_name, tag in DS_TAGS.items():
    pt_p  = os.path.join(MODELS_DIR, f'torch_mlp_{tag}.pt')
    cfg_p = os.path.join(MODELS_DIR, f'torch_mlp_{tag}_cfg.json')
    X_tr, X_te, y_tr, y_te, n_cls = DATA[ds_name]
    n_feat = X_te.shape[1]
    if os.path.exists(pt_p) and os.path.exists(cfg_p):
        with open(cfg_p) as f: c = json.load(f)
        m = TorchMLP(c['n_features'], c['n_classes']).to(DEVICE)
        m.load_state_dict(torch.load(pt_p, map_location=DEVICE)); m.eval()
        TORCH_MLPS[ds_name] = m
        print(f'✓ TorchMLP/{ds_name}: loaded')
    else:
        print(f'Training TorchMLP/{ds_name}...')
        m = train_torch_mlp(X_tr, y_tr, n_feat, n_cls)
        torch.save(m.state_dict(), pt_p)
        with open(cfg_p,'w') as f:
            json.dump({'n_features':n_feat,'n_classes':n_cls}, f)
        TORCH_MLPS[ds_name] = m
        print(f'✓ TorchMLP/{ds_name}: trained')

for ds_name, tag in DS_TAGS.items():
    X_tr, X_te, y_tr, y_te, n_cls = DATA[ds_name]
    n_feat = X_te.shape[1]
    n_samp = min(5000, len(X_te))
    idx    = np.random.choice(len(X_te), n_samp, replace=False)
    Xb, yb = X_te[idx], y_te[idx]
    cnn_surr = MODELS[ds_name].get('CNN')

    # White-box: MLP (via TorchMLP) and CNN
    for mn, model in [('MLP', TORCH_MLPS[ds_name]),
                       ('CNN', MODELS[ds_name].get('CNN'))]:
        if model is None: continue
        art_clf = make_art_clf(model, n_feat, n_cls)
        for atk_name in ['FGSM','PGD','CW']:
            for eps in EPS_LIST:
                key = f'{ds_name}|{mn}|{atk_name}|{eps}'
                if key in wb and 'error' not in wb[key]: continue
                try:
                    if atk_name == 'CW':
                        print(f'C&W: {CW_SAMPLES} samples, {CW_ITERS} iters')
                        idx_cw = np.random.choice(len(X_te), CW_SAMPLES, replace=False)
                        Xc, yc = X_te[idx_cw], y_te[idx_cw]
                        atk = CarliniL2Method(art_clf, confidence=0,
                            learning_rate=0.01, binary_search_steps=9,
                            max_iter=CW_ITERS, batch_size=32)
                        X_adv = atk.generate(x=Xc.astype(np.float32))
                        y_adv = predict_batched(model, X_adv)
                        y_cln = predict_batched(model, Xc)
                        asr = float(1-accuracy_score(yc, y_adv))
                        acc_clean = float(accuracy_score(yc, y_cln))
                        f1_adv = float(f1_score(yc, y_adv, average='macro', zero_division=0))
                    else:
                        if atk_name == 'FGSM':
                            atk = FastGradientMethod(art_clf, eps=eps,
                                eps_step=eps, norm=np.inf, batch_size=512)
                        else:
                            atk = ProjectedGradientDescent(art_clf, eps=eps,
                                eps_step=eps/4, max_iter=40, norm=np.inf, batch_size=256)
                        X_adv = atk.generate(x=Xb.astype(np.float32))
                        y_adv = predict_batched(model, X_adv)
                        y_cln = predict_batched(model, Xb)
                        asr = float(1-accuracy_score(yb, y_adv))
                        acc_clean = float(accuracy_score(yb, y_cln))
                        f1_adv = float(f1_score(yb, y_adv, average='macro', zero_division=0))
                    wb[key] = {'ds':ds_name,'model':mn,'threat':'white-box',
                        'attack':atk_name,'eps':eps,'asr':asr,
                        'acc_clean':acc_clean,'f1_adv':f1_adv}
                    with open(WB_PATH,'w') as f: json.dump(wb,f,indent=2)
                    print(f'→ {ds_name} | {mn} | {atk_name} | eps={eps}')
                    print(f'  ASR={asr:.4f}  acc_clean={acc_clean:.4f} ✓')
                except Exception as e:
                    wb[key] = {'error':str(e)}
                    with open(WB_PATH,'w') as f: json.dump(wb,f,indent=2)
                    print(f'  ERROR: {e}')

    # Black-box: RF and XGBoost via CNN surrogate
    if cnn_surr is None: continue
    cnn_art = make_art_clf(cnn_surr, n_feat, n_cls)
    for mn, tgt in [('RF', MODELS[ds_name].get('RF')),
                     ('XGB', MODELS[ds_name].get('XGB'))]:
        if tgt is None: continue
        # Auto-train RF if missing
        if mn=='RF' and tgt is None:
            print(f'Training RF/{ds_name}...')
            tgt = RandomForestClassifier(n_estimators=100,
                class_weight='balanced', random_state=SEED, n_jobs=-1)
            tgt.fit(X_tr, y_tr)
            joblib.dump(tgt, os.path.join(MODELS_DIR, f'rf_{tag}.joblib'))
            MODELS[ds_name]['RF'] = tgt
        for atk_name in ['FGSM','PGD']:
            for eps in EPS_LIST:
                key = f'{ds_name}|{mn}|{atk_name}|{eps}'
                if key in wb and 'error' not in wb[key]: continue
                try:
                    if atk_name=='FGSM':
                        atk = FastGradientMethod(cnn_art, eps=eps,
                            eps_step=eps, norm=np.inf, batch_size=512)
                    else:
                        atk = ProjectedGradientDescent(cnn_art, eps=eps,
                            eps_step=eps/4, max_iter=40, norm=np.inf, batch_size=256)
                    X_adv = atk.generate(x=Xb.astype(np.float32))
                    y_adv = tgt.predict(X_adv)
                    asr = float(1-accuracy_score(yb, y_adv))
                    acc_clean = float(accuracy_score(yb, tgt.predict(Xb)))
                    f1_adv = float(f1_score(yb, y_adv, average='macro', zero_division=0))
                    wb[key] = {'ds':ds_name,'model':mn,
                        'threat':'black-box (CNN surrogate)',
                        'attack':atk_name,'eps':eps,'asr':asr,
                        'acc_clean':acc_clean,'f1_adv':f1_adv}
                    with open(WB_PATH,'w') as f: json.dump(wb,f,indent=2)
                    print(f'→ {ds_name} | {mn} (black-box) | {atk_name} | eps={eps}')
                    print(f'  ASR={asr:.4f}  acc_clean={acc_clean:.4f} ✓')
                except Exception as e:
                    wb[key] = {'error':str(e)}
                    with open(WB_PATH,'w') as f: json.dump(wb,f,indent=2)
                    print(f'  ERROR: {e}')

valid = sum(1 for v in wb.values() if 'error' not in v)
print(f'\n✓ White-box done: {valid}/120 saved.')

## CELL 6 — Cross-architecture transfer (144 combinations)
2 src × 3 tgt × 2 atk × 3 eps × 4 datasets = 144

In [ ]:
CA_PATH = os.path.join(SEC7_DIR, 'ca_results_v2.json')
ca = json.load(open(CA_PATH)) if os.path.exists(CA_PATH) else {}
wb = json.load(open(os.path.join(SEC7_DIR,'wb_results_v2.json')))
valid = sum(1 for v in ca.values() if 'error' not in v)
print(f'Cross-arch: {valid}/144 already computed.')

SRC_NEURAL = ['MLP','CNN']
TGT_ALL    = ['RF','XGB','MLP','CNN']

for ds_name, tag in DS_TAGS.items():
    X_tr, X_te, y_tr, y_te, n_cls = DATA[ds_name]
    n_feat = X_te.shape[1]
    n_samp = min(3000, len(X_te))
    idx    = np.random.choice(len(X_te), n_samp, replace=False)
    Xb, yb = X_te[idx], y_te[idx]

    for src_mn in SRC_NEURAL:
        src_model = (TORCH_MLPS[ds_name] if src_mn=='MLP'
                     else MODELS[ds_name].get('CNN'))
        if src_model is None: continue
        src_art = make_art_clf(src_model, n_feat, n_cls)

        for atk_name in ['FGSM','PGD']:
            for eps in EPS_LIST:
                try:
                    if atk_name=='FGSM':
                        atk = FastGradientMethod(src_art, eps=eps,
                            eps_step=eps, norm=np.inf, batch_size=512)
                    else:
                        atk = ProjectedGradientDescent(src_art, eps=eps,
                            eps_step=eps/4, max_iter=40, norm=np.inf, batch_size=256)
                    X_adv = atk.generate(x=Xb.astype(np.float32))
                except Exception as e:
                    print(f'  Gen error {src_mn}/{atk_name}/{eps}: {e}'); continue

                wb_key = f'{ds_name}|{src_mn}|{atk_name}|{eps}'
                wb_asr = wb.get(wb_key,{}).get('asr', None)

                for tgt_mn in TGT_ALL:
                    if tgt_mn == src_mn: continue
                    key = f'{ds_name}|{src_mn}→{tgt_mn}|{atk_name}|{eps}'
                    if key in ca and 'error' not in ca[key]: continue
                    try:
                        if tgt_mn=='MLP':   tgt_model = TORCH_MLPS[ds_name]
                        elif tgt_mn=='CNN': tgt_model = MODELS[ds_name].get('CNN')
                        elif tgt_mn=='RF':  tgt_model = MODELS[ds_name].get('RF')
                        else:               tgt_model = MODELS[ds_name].get('XGB')
                        if tgt_model is None: continue

                        if tgt_mn in ['MLP','CNN']:
                            y_pred = predict_batched(tgt_model, X_adv)
                        else:
                            y_pred = tgt_model.predict(X_adv)

                        asr = float(1-accuracy_score(yb, y_pred))
                        tr  = float(asr/wb_asr) if wb_asr and wb_asr>0 else None

                        ca[key] = {'ds':ds_name,'src':src_mn,'tgt':tgt_mn,
                            'attack':atk_name,'eps':eps,'asr':asr,'tr':tr}
                        with open(CA_PATH,'w') as f: json.dump(ca,f,indent=2)

                        # FIX: окрема змінна для TR щоб уникнути помилки форматування
                        tr_str = f'{tr:.3f}' if tr is not None else 'N/A'
                        print(f'{src_mn}→{tgt_mn} {atk_name} eps={eps} | '
                              f'ASR={asr:.3f} TR={tr_str}')
                    except Exception as e:
                        ca[key] = {'error':str(e)}
                        with open(CA_PATH,'w') as f: json.dump(ca,f,indent=2)

valid = sum(1 for v in ca.values() if 'error' not in v)
print(f'\n✓ Cross-arch done: {valid}/144 saved.')

## CELL 7 — Cross-dataset transfer via aligned features (36 combinations)
**Правильний протокол:** delta = X_adv_src - X_src → застосовуємо до TARGET samples

In [ ]:
CD_PATH = os.path.join(SEC7_DIR, 'cd_results_v2.json')
cd = json.load(open(CD_PATH)) if os.path.exists(CD_PATH) else {}
valid = sum(1 for v in cd.values() if 'error' not in v)
print(f'Cross-dataset: {valid}/36 already computed.')

# Aligned models: навчені на перших N_AL ознаках кожного датасету
AL_MODELS = {}
AL_DATA   = {}

for ds_name, tag in [('CIC-IDS 2017','CIC-IDS_2017'),
                      ('UNSW-NB15','UNSW-NB15')]:
    X_tr, X_te, y_tr, y_te, n_cls = DATA[ds_name]
    X_tr_al = X_tr[:, :N_AL]
    X_te_al = X_te[:, :N_AL]
    AL_DATA[ds_name]   = (X_tr_al, X_te_al, y_tr, y_te, n_cls)
    AL_MODELS[ds_name] = {}

    for mn in ['RF','XGB','MLP']:
        pt_p  = os.path.join(MODELS_DIR, f'aligned_mlp_{tag}.pt')
        cfg_p = os.path.join(MODELS_DIR, f'aligned_mlp_{tag}_cfg.json')
        sk_p  = os.path.join(MODELS_DIR, f'aligned_{mn.lower()}_{tag}.joblib')

        if mn == 'MLP':
            if os.path.exists(pt_p) and os.path.exists(cfg_p):
                with open(cfg_p) as f: c = json.load(f)
                m = TorchMLP(c['n_features'], c['n_classes']).to(DEVICE)
                m.load_state_dict(torch.load(pt_p, map_location=DEVICE))
                m.eval(); AL_MODELS[ds_name][mn] = m
                print(f'✓ Aligned MLP/{ds_name}: loaded')
            else:
                print(f'Training aligned MLP/{ds_name}...')
                m = train_torch_mlp(X_tr_al, y_tr, N_AL, n_cls)
                torch.save(m.state_dict(), pt_p)
                with open(cfg_p,'w') as f:
                    json.dump({'n_features':N_AL,'n_classes':n_cls}, f)
                AL_MODELS[ds_name][mn] = m
                print(f'✓ Aligned MLP/{ds_name}: trained')
        else:
            if os.path.exists(sk_p):
                AL_MODELS[ds_name][mn] = joblib.load(sk_p)
                print(f'✓ Aligned {mn}/{ds_name}: loaded')
            else:
                print(f'Training aligned {mn}/{ds_name}...')
                if mn=='RF':
                    model = RandomForestClassifier(n_estimators=100,
                        class_weight='balanced', random_state=SEED, n_jobs=-1)
                else:
                    obj = 'multi:softprob' if n_cls>2 else 'binary:logistic'
                    model = XGBClassifier(n_estimators=100, max_depth=6,
                        learning_rate=0.1, random_state=SEED, objective=obj,
                        num_class=n_cls if n_cls>2 else None,
                        verbosity=0, n_jobs=-1)
                model.fit(X_tr_al, y_tr)
                joblib.dump(model, sk_p)
                AL_MODELS[ds_name][mn] = model
                print(f'✓ Aligned {mn}/{ds_name}: trained')

# Правильний протокол: delta transfer
DIRECTIONS = [('CIC-IDS 2017','UNSW-NB15'), ('UNSW-NB15','CIC-IDS 2017')]

for src_ds, tgt_ds in DIRECTIONS:
    X_tr_s, X_te_s, y_tr_s, y_te_s, n_cls_s = AL_DATA[src_ds]
    X_tr_t, X_te_t, y_tr_t, y_te_t, n_cls_t = AL_DATA[tgt_ds]

    n_src = min(3000, len(X_te_s))
    idx_s = np.random.choice(len(X_te_s), n_src, replace=False)
    Xs    = X_te_s[idx_s]

    n_tgt = min(3000, len(X_te_t))
    idx_t = np.random.choice(len(X_te_t), n_tgt, replace=False)
    Xt, yt = X_te_t[idx_t], y_te_t[idx_t]

    for mn in ['RF','XGB','MLP']:
        src_model = AL_MODELS[src_ds].get(mn)
        tgt_model = AL_MODELS[tgt_ds].get(mn)
        if src_model is None or tgt_model is None: continue

        # Surrogate: MLP для RF/XGB, сам для MLP
        if mn=='MLP': src_art = make_art_clf(src_model, N_AL, n_cls_s)
        else:
            mlp_surr = AL_MODELS[src_ds].get('MLP')
            if mlp_surr is None: continue
            src_art = make_art_clf(mlp_surr, N_AL, n_cls_s)

        for atk_name in ['FGSM','PGD']:
            for eps in EPS_LIST:
                key = f'{src_ds}→{tgt_ds}|{mn}|{atk_name}|{eps}'
                if key in cd and 'error' not in cd[key]: continue
                try:
                    if atk_name=='FGSM':
                        atk = FastGradientMethod(src_art, eps=eps,
                            eps_step=eps, norm=np.inf, batch_size=512)
                    else:
                        atk = ProjectedGradientDescent(src_art, eps=eps,
                            eps_step=eps/4, max_iter=40, norm=np.inf, batch_size=256)

                    # 1. Генеруємо adversarial на SOURCE
                    X_adv_src = atk.generate(x=Xs.astype(np.float32))
                    # 2. Обчислюємо delta
                    delta = X_adv_src - Xs
                    # 3. Застосовуємо delta до TARGET
                    n_apply = min(len(delta), len(Xt))
                    X_tgt_perturbed = np.clip(
                        Xt[:n_apply] + delta[:n_apply], 0.0, 1.0
                    ).astype(np.float32)
                    yt_eval = yt[:n_apply]

                    # 4. Оцінюємо TARGET model
                    if mn=='MLP':
                        y_pred  = predict_batched(tgt_model, X_tgt_perturbed)
                        y_clean = predict_batched(tgt_model, Xt[:n_apply])
                    else:
                        y_pred  = tgt_model.predict(X_tgt_perturbed)
                        y_clean = tgt_model.predict(Xt[:n_apply])

                    asr_tgt   = float(1-accuracy_score(yt_eval, y_pred))
                    acc_clean = float(accuracy_score(yt_eval, y_clean))

                    cd[key] = {'src_ds':src_ds,'tgt_ds':tgt_ds,'model':mn,
                        'attack':atk_name,'eps':eps,'asr_tgt':asr_tgt,
                        'acc_clean_tgt':acc_clean,'n_samples':n_apply}
                    with open(CD_PATH,'w') as f: json.dump(cd,f,indent=2)
                    print(f'{src_ds[:10]}→{tgt_ds[:10]} | {mn:4s} | '
                          f'{atk_name} eps={eps} | '
                          f'ASR_tgt={asr_tgt:.4f} clean={acc_clean:.4f} ✓')
                except Exception as e:
                    cd[key] = {'error':str(e)}
                    with open(CD_PATH,'w') as f: json.dump(cd,f,indent=2)
                    print(f'  ERROR: {e}')

valid = sum(1 for v in cd.values() if 'error' not in v)
print(f'\n✓ Cross-dataset done: {valid}/36 saved.')

## CELL 8 — Statistical tests + Robustness Score
**Wilcoxon на matched triplets** (тільки MLP+CNN де є C&W), не на всіх 120 значеннях.

In [ ]:
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

STAT_PATH = os.path.join(SEC7_DIR, 'stat_results_v2.json')

if os.path.exists(STAT_PATH):
    stat_results = json.load(open(STAT_PATH))
    print('✓ Statistical tests loaded from Drive')
    for k,v in stat_results.get('wilcoxon',{}).items():
        sig = '*' if v.get('significant') else 'ns'
        print(f"  {k}: W={v['statistic']:.1f} p={v['p_value']:.4f} {sig}")
else:
    wb = json.load(open(os.path.join(SEC7_DIR,'wb_results_v2.json')))
    valid_wb = {k:v for k,v in wb.items() if 'error' not in v}

    def to_python(obj):
        if isinstance(obj,(np.integer,)): return int(obj)
        if isinstance(obj,(np.floating,)): return float(obj)
        return obj

    # Robustness Score
    rs_data = {}
    for k,v in valid_wb.items():
        rs = v['acc_clean'] * (1-v['asr'])
        rs_data[k] = {'ds':v['ds'],'model':v['model'],
            'attack':v['attack'],'eps':v['eps'],'rs':rs}

    # Wilcoxon: matched triplets (тільки комбінації де є всі 3 атаки)
    grouped = defaultdict(dict)
    for k,v in valid_wb.items():
        grp_key = f"{v['ds']}|{v['model']}|{v['eps']}"
        grouped[grp_key][v['attack']] = v['asr']

    matched = [
        (g['FGSM'],g['PGD'],g['CW'])
        for g in grouped.values()
        if all(a in g for a in ['FGSM','PGD','CW'])
    ]
    print(f'Matched triplets: {len(matched)} (MLP+CNN × 4 datasets × 3 epsilon)')

    fgsm_m = np.array([x[0] for x in matched])
    pgd_m  = np.array([x[1] for x in matched])
    cw_m   = np.array([x[2] for x in matched])

    pairs = [('FGSM_vs_PGD',fgsm_m,pgd_m),
             ('FGSM_vs_CW', fgsm_m,cw_m),
             ('PGD_vs_CW',  pgd_m, cw_m)]

    raw = {}
    for name,a,b in pairs:
        w,p = wilcoxon(a,b)
        raw[name] = {'statistic':float(w),'p_value':float(p)}

    pvals = [raw[k]['p_value'] for k in raw]
    reject,pvals_corr,_,_ = multipletests(pvals, method='fdr_bh')

    wilcoxon_res = {}
    for i,(k,v) in enumerate(raw.items()):
        wilcoxon_res[k] = {**v,
            'p_corrected':float(pvals_corr[i]),
            'significant':bool(reject[i]),
            'n_triplets':len(matched),
            'method':'matched triplets (MLP+CNN only)'}

    stat_results = {'wilcoxon':wilcoxon_res,'robustness_scores':rs_data}
    with open(STAT_PATH,'w') as f:
        json.dump(stat_results,f,indent=2,default=to_python)

    print('\n✓ Statistical tests saved.')
    for k,v in wilcoxon_res.items():
        sig = '*' if v['significant'] else 'ns'
        print(f"  {k}: W={v['statistic']:.1f} "
              f"p={v['p_value']:.4f} p_corr={v['p_corrected']:.4f} {sig}")

## CELL 9 — Feature Squeezing defense (16 combinations)

In [ ]:
DEF_PATH = os.path.join(SEC7_DIR, 'defense_results_v2.json')

if os.path.exists(DEF_PATH):
    def_results = json.load(open(DEF_PATH))
    print(f'✓ Defense results loaded: {len(def_results)} entries')
    for k,v in def_results.items():
        print(f"  {v['ds']:20s}|{v['model']:8s}|{v['attack']:4s} "
              f"clean={v['acc_clean']:.3f} "
              f"ASR before={v['asr_before']:.3f} "
              f"after FS={v['asr_fs']:.3f} "
              f"clean_fs={v['clean_fs']:.3f}")
else:
    def feature_squeeze(X, bits=4, window=3):
        step = 1.0/(2**bits-1)
        X_sq = np.round(X/step)*step
        X_sq = np.array([np.convolve(row,np.ones(window)/window,'same')
                         for row in X_sq])
        return np.clip(X_sq,0,1).astype(np.float32)

    wb = json.load(open(os.path.join(SEC7_DIR,'wb_results_v2.json')))
    def_results = {}
    EPS_DEF = 0.05

    for ds_name in ['CIC-IDS 2017','UNSW-NB15']:
        X_tr,X_te,y_tr,y_te,n_cls = DATA[ds_name]
        n_feat = X_te.shape[1]
        n_samp = min(3000,len(X_te))
        idx = np.random.choice(len(X_te),n_samp,replace=False)
        Xb,yb = X_te[idx],y_te[idx]

        for mn in ['CNN','MLP','RF','XGBoost']:
            for atk_name in ['FGSM','PGD']:
                key_wb = f'{ds_name}|{mn}|{atk_name}|{EPS_DEF}'
                if key_wb not in wb or 'error' in wb[key_wb]: continue

                if mn in ['MLP','CNN']:
                    model = TORCH_MLPS[ds_name] if mn=='MLP' else MODELS[ds_name].get('CNN')
                    if model is None: continue
                    art_clf = make_art_clf(model,n_feat,n_cls)
                else:
                    cnn_s = MODELS[ds_name].get('CNN')
                    if cnn_s is None: continue
                    art_clf = make_art_clf(cnn_s,n_feat,n_cls)
                    model = MODELS[ds_name].get('RF' if mn=='RF' else 'XGB')
                    if model is None: continue

                try:
                    if atk_name=='FGSM':
                        atk = FastGradientMethod(art_clf,eps=EPS_DEF,
                            eps_step=EPS_DEF,norm=np.inf,batch_size=512)
                    else:
                        atk = ProjectedGradientDescent(art_clf,eps=EPS_DEF,
                            eps_step=EPS_DEF/4,max_iter=40,norm=np.inf,batch_size=256)

                    X_adv     = atk.generate(x=Xb.astype(np.float32))
                    X_adv_fs  = feature_squeeze(X_adv)
                    X_clean_fs= feature_squeeze(Xb)

                    if mn in ['MLP','CNN']:
                        p_adv    = predict_batched(model,X_adv)
                        p_adv_fs = predict_batched(model,X_adv_fs)
                        p_cln_fs = predict_batched(model,X_clean_fs)
                        p_clean  = predict_batched(model,Xb)
                    else:
                        p_adv    = model.predict(X_adv)
                        p_adv_fs = model.predict(X_adv_fs)
                        p_cln_fs = model.predict(X_clean_fs)
                        p_clean  = model.predict(Xb)

                    asr_before = float(1-accuracy_score(yb,p_adv))
                    asr_fs     = float(1-accuracy_score(yb,p_adv_fs))
                    acc_clean  = float(accuracy_score(yb,p_clean))
                    clean_fs   = float(accuracy_score(yb,p_cln_fs))

                    def_key = f'{ds_name}|{mn}|{atk_name}'
                    def_results[def_key] = {'ds':ds_name,'model':mn,
                        'attack':atk_name,'acc_clean':acc_clean,
                        'asr_before':asr_before,'asr_fs':asr_fs,'clean_fs':clean_fs}
                    with open(DEF_PATH,'w') as f: json.dump(def_results,f,indent=2)
                    print(f"{ds_name}|{mn}|{atk_name}: "
                          f"clean={acc_clean:.3f} "
                          f"ASR before={asr_before:.3f} "
                          f"after FS={asr_fs:.3f} "
                          f"clean_fs={clean_fs:.3f} ✓")
                except Exception as e:
                    print(f'  ERROR {key_wb}: {e}')

    print(f'\n✓ Defense done: {len(def_results)} combinations saved.')

## CELL 10 — Final Status

In [ ]:
print('=== SECTION 7 v2 FINAL STATUS ===')
# Правильні очікувані кількості:
# Cross-arch: 144 (2 src × 3 tgt × 2 atk × 3 eps × 4 ds)
expected = {
    'White-box results'     : ('wb_results_v2.json',     120),
    'Cross-arch results'    : ('ca_results_v2.json',     144),
    'Cross-dataset results' : ('cd_results_v2.json',      36),
    'Statistical tests'     : ('stat_results_v2.json',     2),
    'Defense results'       : ('defense_results_v2.json', 16),
}
all_ok = True
for label,(fname,exp) in expected.items():
    p = os.path.join(SEC7_DIR, fname)
    if os.path.exists(p):
        data  = json.load(open(p))
        valid = sum(1 for v in data.values() if 'error' not in v)
        kb    = os.path.getsize(p)/1024
        ok    = valid >= exp
        icon  = '✓' if ok else '⚠'
        print(f'  {icon}  {label:<30} {kb:>6.0f} KB  ({valid}/{exp} entries)')
        if not ok: all_ok = False
    else:
        print(f'  ✗  {label} — MISSING')
        all_ok = False
print()
if all_ok:
    print('✓ All Section 7 v2 data ready. Run Section7_Figures_v2.ipynb next.')
else:
    print('⚠ Some files missing — rerun corresponding CELL above.')